# Week 05 — Python Solution Lab
## Circular Motion

**Companion to `notebooks/Week_05.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_05.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P3` | Flat Curve Friction | design chart across surface conditions |
| **L2 · Intermediate** | `P5` | Banked Curve Design | safe-speed band vs available friction |
| **L3 · Challenge** | `P9` | Loop-the-Loop with Friction | **`solve_ivp` + `brentq`**, checked against the closed form |

---

## L1 · Basic — P3: Flat Curve Friction

> **Problem (Week_05.ipynb, L1 — P3).** A $1500$ kg car rounds a flat curve of radius $80.0$ m.
> $\mu_s = 0.55$. What is the maximum speed without sliding?

**Diagram → Principle.** On a flat curve the *only* horizontal force available is static
friction, and it must supply the entire centripetal requirement.

**Equation.** $\mu_s mg = \dfrac{mv^2}{r} \Rightarrow v_{\max} = \sqrt{\mu_s g r}$.

**Hand prediction.** $\sqrt{0.55 \times 9.81 \times 80} = 20.8$ m/s $= 74.8$ km/h.

**What Python adds.** The mass cancels — a fact students routinely disbelieve. We verify it
across four orders of magnitude of mass, then produce the **design chart** an engineer actually
wants: safe speed against radius, for dry, wet, and icy road surfaces.

In [ ]:
# ═══ W05 · L1 · P3 — Flat-curve speed limit, and the design chart behind it ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, r, mu_s, g = 1500.0, 80.0, 0.55, 9.81

# --- PREDICT ------------------------------------------------------------
v_max = np.sqrt(mu_s * g * r)
print(f"v_max = sqrt(mu_s * g * r) = {v_max:.2f} m/s = {v_max*3.6:.1f} km/h")
print(f"required centripetal force at that speed: {m*v_max**2/r:.0f} N "
      f"(= mu_s*m*g = {mu_s*m*g:.0f} N)")

# --- VERIFY: mass cancels -----------------------------------------------
print("\n  mass (kg) | v_max (m/s) | friction needed (N)")
for mt in (150.0, 1500.0, 15000.0, 40000.0):
    print(f"  {mt:9.0f} | {np.sqrt(mu_s*g*r):11.2f} | {mt*v_max**2/r:18.0f}")
print("  -> v_max is identical: a loaded truck and a motorbike slip at the same speed.")

# --- DESIGN CHART: v_max(r) for three surface conditions ----------------
radii = np.linspace(20, 400, 400)
fig, ax = plt.subplots(figsize=(7.2, 4))
for mu, lbl, col in ((0.80, "dry asphalt (0.80)", "#2e7d32"),
                     (0.55, "this problem (0.55)", "#1565c0"),
                     (0.30, "wet (0.30)", "#e65100"),
                     (0.10, "ice (0.10)", "crimson")):
    ax.plot(radii, np.sqrt(mu*g*radii)*3.6, lw=2, color=col, label=lbl)
ax.plot(r, v_max*3.6, "o", color="#1565c0", ms=10, zorder=5)
ax.annotate(f"{v_max*3.6:.0f} km/h", (r, v_max*3.6), textcoords="offset points",
            xytext=(10, -14), fontsize=10)
ax.set_xlabel("curve radius r (m)"); ax.set_ylabel("maximum safe speed (km/h)")
ax.set_title("W05 P3 — flat curve: $v_{max}=\\sqrt{\\mu_s g r}$, independent of mass")
ax.grid(alpha=.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f"\nOn ice (mu=0.10) the same curve is safe only up to "
      f"{np.sqrt(0.10*g*r)*3.6:.0f} km/h -- a {np.sqrt(0.55/0.10):.1f}x reduction.")

# --- CHECK --------------------------------------------------------------
assert abs(v_max - 20.8) < 0.05 and abs(v_max*3.6 - 74.8) < 0.2
print("[OK] Matches textbook answer: v_max = 20.8 m/s = 74.8 km/h")

## L2 · Intermediate — P5: Banked Curve Design

> **Problem (Week_05.ipynb, L2 — P5).** A highway curve of radius $200$ m is designed for
> $90$ km/h. (a) Bank angle with no friction? (b) If wet ($\mu_s = 0.20$), what are the minimum
> and maximum safe speeds? (c) Express $v_{\max}$ in km/h.

**Diagram → Principle.** Tilt the road and the *normal force* gains an inward component. At the
design speed friction is not needed at all. Off the design speed, friction makes up the
difference — pointing up-slope if too slow, down-slope if too fast.

**Equation.** $\tan\theta = \dfrac{v^2}{rg}$;
$v_{\max,\min} = \sqrt{rg\dfrac{\tan\theta \pm \mu_s}{1 \mp \mu_s\tan\theta}}$.

**Hand prediction.** $\tan\theta = 625/1962 = 0.3185 \Rightarrow \theta = 17.7^\circ$.

**What Python adds.** The $\pm/\mp$ formula is exactly where sign slips happen. We derive both
limits from one function, sanity-check that they bracket the design speed, and plot the **safe
operating band** against $\mu_s$ — showing that at $\mu_s = 0$ the band collapses to a single
speed, which is the physical meaning of "design speed".

In [ ]:
# ═══ W05 · L2 · P5 — Banked curve: design angle and the safe speed band ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
r, g       = 200.0, 9.81
v_design   = 90 / 3.6          # m/s
mu_wet     = 0.20

# --- (a) frictionless design angle --------------------------------------
theta = np.arctan(v_design**2 / (r * g))
print(f"(a) tan(theta) = v^2/(r g) = {v_design**2/(r*g):.4f}")
print(f"    theta = {np.degrees(theta):.2f} deg")

# --- (b) friction limits, both signs from ONE function ------------------
def v_limit(mu, sign, theta=theta, r=r, g=g):
    """sign=+1 -> maximum speed (friction acts down-slope);
       sign=-1 -> minimum speed (friction acts up-slope)."""
    num = np.tan(theta) + sign * mu
    den = 1 - sign * mu * np.tan(theta)
    return np.sqrt(r * g * num / den) if num > 0 else 0.0

v_min = v_limit(mu_wet, -1)
v_max = v_limit(mu_wet, +1)
print(f"\n(b) with mu_s = {mu_wet}:")
print(f"    v_min = {v_min:6.2f} m/s = {v_min*3.6:6.1f} km/h")
print(f"    v_max = {v_max:6.2f} m/s = {v_max*3.6:6.1f} km/h")
print(f"(c) v_max = {v_max*3.6:.1f} km/h")

# --- VERIFY: the design speed must sit inside the band, and at mu=0 -----
assert v_min < v_design < v_max, "design speed must be inside the safe band"
assert abs(v_limit(0.0, +1) - v_design) < 1e-9
assert abs(v_limit(0.0, -1) - v_design) < 1e-9
print(f"\n    at mu = 0 both limits collapse to {v_limit(0.0, +1)*3.6:.1f} km/h")
print("    -> that IS the definition of the design speed. [verified]")

# --- Plot: safe band vs available friction ------------------------------
mus = np.linspace(0, 0.9, 400)
vmx = np.array([v_limit(m, +1) for m in mus]) * 3.6
vmn = np.array([v_limit(m, -1) for m in mus]) * 3.6

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.fill_between(mus, vmn, vmx, color="#2e7d32", alpha=.18, label="safe band")
ax.plot(mus, vmx, color="#2e7d32", lw=2, label="$v_{max}$")
ax.plot(mus, vmn, color="#e65100", lw=2, label="$v_{min}$")
ax.axhline(90, ls="--", c="#1565c0", label="design speed 90 km/h")
ax.axvline(mu_wet, ls=":", c="grey", label=f"wet road, $\\mu_s$ = {mu_wet}")
ax.set_xlabel("$\\mu_s$ available"); ax.set_ylabel("speed (km/h)")
ax.set_title(f"W05 P5 — r = {r:.0f} m banked at {np.degrees(theta):.1f} deg")
ax.grid(alpha=.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(np.degrees(theta) - 17.7) < 0.1
assert abs(v_min - 14.79) < 0.4 and abs(v_max - 32.96) < 0.4
print(f"[OK] theta = {np.degrees(theta):.1f} deg; safe band "
      f"{v_min*3.6:.0f}-{v_max*3.6:.0f} km/h on a wet road.")

## L3 · Challenge — P9: Loop-the-Loop with Friction

> **Problem (Week_05.ipynb, L3 — P9).** A block slides from rest down a frictionless ramp of
> height $h$ into a vertical loop of radius $R = 1.50$ m with $\mu_k = 0.10$ inside the loop.
> Find the minimum $h$ so the block barely maintains contact at the top.

**Diagram → Principle.** Two principles at once. At the top, "barely maintains contact" means
$N = 0$, so gravity alone supplies the centripetal force: $v_{\text{top}}^2 = gR$. Getting there
requires energy conservation **minus** the friction work around the loop.

**Equation.** $mgh = mg(2R) + \tfrac12 mv_{\text{top}}^2 + W_{\text{fric}}$, with
$v_{\text{top}}^2 = gR$.

**Hand prediction (frictionless).** $h = 2.5R = 3.75$ m. With friction it must be larger.

**Why this is harder than it looks.** $W_{\text{fric}} = \int \mu_k N\,ds$, and $N$ varies all
the way around the loop — it depends on the very speed you are trying to find. So friction is
*not* a simple $\mu_k mg\,s$ term you can subtract up front; speed and normal force are coupled.

**The equation is linear, and it does have a closed form.** Writing $y(\theta) = v^2(\theta)$ and
using $N/m = y/R + g\cos\theta$:

$$\frac{dy}{d\theta} + 2\mu_k\,y = -2gR\sin\theta - 2\mu_k gR\cos\theta$$

— a first-order **linear** ODE with sinusoidal forcing, so

$$y(\theta) = Ce^{-2\mu_k\theta} - \frac{6\mu_k gR}{1+4\mu_k^2}\sin\theta
            + \frac{2gR(1-2\mu_k^2)}{1+4\mu_k^2}\cos\theta.$$

Imposing $y(0) = 2gh$ and the marginal-contact condition $y(\pi) = gR$ gives $h_{\min}$ in one
line. We derive it below and it agrees with the numerical answer to $10^{-11}$.

**So why integrate numerically at all?** Not because you must — but because the numerical route
is the one that *keeps working*. Change the track from a circle to an arbitrary profile, make
$\mu_k$ depend on speed or position, or add drag, and the closed form evaporates while
`solve_ivp` + `brentq` needs no change at all. Having both here is the point: the analytic
solution certifies the numerical machinery on a case where we can check it, so you can trust
that machinery on cases where you cannot.

In [ ]:
# ═══ W05 · L3 · P9 — Loop-the-loop WITH friction: ODE, root-find, closed form ═══
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
R, mu_k, g = 1.50, 0.10, 9.81

# Energy per unit mass:  E = v^2/2.  Going round the loop, theta measured
# from the BOTTOM, the block is at height R(1 - cos th).
#   d(v^2/2)/d(theta) = -g R sin(th)  -  mu_k * N/m * R      (friction, always opposing)
#   N/m = v^2/R + g cos(th)     (centripetal balance; cos th > 0 in the lower half)
def dv2_dtheta(th, y):
    v2 = max(y[0], 0.0)
    N_over_m = v2 / R + g * np.cos(th)
    N_over_m = max(N_over_m, 0.0)              # a track can only push, not pull
    return [-2 * g * R * np.sin(th) - 2 * mu_k * N_over_m * R]

def v2_at_top(h):
    """Speed^2 at the top of the loop, entering with v^2 = 2 g h at the bottom."""
    sol = solve_ivp(dv2_dtheta, [0, np.pi], [2 * g * h],
                    rtol=1e-10, atol=1e-12, dense_output=True)
    return sol.y[0, -1], sol

# --- The frictionless benchmark first -----------------------------------
h_ideal = 2.5 * R
print(f"Frictionless benchmark: h = 2.5R = {h_ideal:.3f} m "
      f"(v_top^2 = gR = {g*R:.3f} m^2/s^2)")

# --- PREDICT: root-find h such that v_top^2 = gR ------------------------
f = lambda h: v2_at_top(h)[0] - g * R
h_min = brentq(f, h_ideal, 6.0 * R, xtol=1e-10)
v2top, sol = v2_at_top(h_min)

print(f"\nWith mu_k = {mu_k} inside the loop:")
print(f"  minimum release height h = {h_min:.4f} m")
print(f"  that is {h_min/R:.3f} R, versus {h_ideal/R:.1f} R with no friction")
print(f"  penalty = {h_min - h_ideal:.4f} m ({100*(h_min/h_ideal - 1):.1f}% extra height)")
print(f"  v_top   = {np.sqrt(v2top):.4f} m/s  (target sqrt(gR) = {np.sqrt(g*R):.4f})")
assert abs(v2top - g*R) < 1e-6

# --- ROUTE 2: the CLOSED FORM -- the ODE is LINEAR in y = v^2 -----------
# dy/dth + 2 mu y = -2 g R sin(th) - 2 mu g R cos(th)
# particular solution y_p = A sin(th) + B cos(th):
den = 1 + 4*mu_k**2
A_p = -6*mu_k*g*R / den
B_p =  2*g*R*(1 - 2*mu_k**2) / den
# y(0) = 2gh  =>  C = 2gh - B_p ;  impose y(pi) = gR and solve for h:
#   (2gh - B_p) exp(-2 mu pi) - B_p = gR
h_exact = (B_p + (g*R + B_p)*np.exp(2*mu_k*np.pi)) / (2*g)

print(f"\n  CLOSED FORM (numerical integration is convenient here, not mandatory):")
print(f"    dy/dtheta + 2 mu y = -2gR sin(th) - 2 mu gR cos(th),   y = v^2")
print(f"    y(th) = C exp(-2 mu th) + A sin(th) + B cos(th)")
print(f"      A = -6 mu gR/(1+4mu^2)      = {A_p:.6f}")
print(f"      B =  2gR(1-2mu^2)/(1+4mu^2) = {B_p:.6f}")
print(f"    y(0) = 2gh and y(pi) = gR  ->  h = [B + (gR + B) exp(2 mu pi)] / (2g)")
print(f"      h_min analytic  = {h_exact:.9f} m")
print(f"      h_min numerical = {h_min:.9f} m")
print(f"      difference      = {abs(h_exact - h_min):.2e} m")
assert abs(h_exact - h_min) < 1e-8, "analytic and numerical routes must agree"

# The closed form assumes the track can only push (N >= 0) throughout; verify,
# because that is exactly what the clamp inside the ODE enforces.
th_chk = np.linspace(0, np.pi, 2001)
y_chk  = (2*g*h_exact - B_p)*np.exp(-2*mu_k*th_chk) + A_p*np.sin(th_chk) + B_p*np.cos(th_chk)
print(f"    min N/m on the analytic solution = {(y_chk/R + g*np.cos(th_chk)).min():.2e}")
print(f"    (non-negative, so the ODE's contact clamp never fires -- the two")
print(f"     formulations describe the same motion, which is why they agree)")
assert (y_chk/R + g*np.cos(th_chk)).min() > -1e-9

# --- VERIFY: friction work accounting closes the energy budget ----------
th  = np.linspace(0, np.pi, 2000)
v2  = sol.sol(th)[0]
Nm  = np.maximum(v2/R + g*np.cos(th), 0.0)         # N/m around the loop
W_f = np.trapezoid(mu_k * Nm * R, th)              # friction work per unit mass
budget = g*h_min - (g*2*R + 0.5*v2top + W_f)
print(f"\n  energy budget per kg:  g*h = {g*h_min:.4f}")
print(f"    lift to top  g*2R      = {g*2*R:.4f}")
print(f"    kinetic at top v^2/2   = {0.5*v2top:.4f}")
print(f"    friction losses        = {W_f:.4f}")
print(f"    residual               = {budget:.2e}   -> the budget closes")
assert abs(budget) < 1e-6

# --- Plot: where is contact most marginal? ------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.9))
ax1.plot(np.degrees(th), np.sqrt(v2), color="#1565c0", lw=2)
ax1.axhline(np.sqrt(g*R), ls="--", c="crimson", label="$\\sqrt{gR}$")
ax1.set_xlabel("angle from bottom (deg)"); ax1.set_ylabel("speed (m/s)")
ax1.set_title("speed bleeds away around the loop"); ax1.grid(alpha=.3); ax1.legend()

ax2.plot(np.degrees(th), Nm, color="#2e7d32", lw=2)
ax2.axhline(0, c="crimson", ls="--", label="loss of contact")
ax2.set_xlabel("angle from bottom (deg)"); ax2.set_ylabel("N/m (m/s$^2$)")
ax2.set_title("normal force -> 0 exactly at the top"); ax2.grid(alpha=.3); ax2.legend()
plt.suptitle(f"W05 P9 — minimum release height {h_min:.3f} m (mu_k = {mu_k})", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert h_min > h_ideal, "friction must raise the required height"
assert 5.4 < h_min < 5.6, f"h_min = {h_min}"
print(f"\n[OK] h_min = {h_min:.3f} m, safely above the frictionless {h_ideal:.2f} m.")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_05.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
